# LoRA by hand, compared against full fine-tuning

This notebook implements LoRA (notes.md §4.1) from scratch in PyTorch, without
the PEFT library, and checks three claims from the notes:

1. A frozen network can be adapted to a shifted task by training only a rank-r
   correction `W' = W + (alpha/r) * B @ A`, where `B` has shape (d_out, r) and
   starts at zero, and `A` has shape (r, d_in) and starts random.
2. With a small fraction of the trainable parameters, this matches full
   fine-tuning on the shifted task.
3. Because `B = 0` at the start, the wrapped model initially computes exactly
   the base model's function; after training, adding `(alpha/r) * B @ A` into
   `W` (merging) gives the same outputs as the adapted model.

The toy task: classify 2-D points as inside or outside a ring (annulus). The
shifted task applies a fixed rotation and stretch to the inputs. The features
the base network learned are still the right ones; only the mapping from
inputs to those features changes. This is the situation in which LoRA is
expected to work, and the notes argue that fine-tuning a pretrained LM is
usually in the same situation.

Runs on CPU in under a minute:
`uv run jupyter execute ml/07_llm_stack/lora_scratch.ipynb`.

## Setup

Imports and a fixed seed, so every run gives the same numbers.

In [1]:
import torch
import torch.nn as nn

torch.manual_seed(0)

## The data

Each point is drawn from a 2-D standard normal and labeled 1 if its radius is
between 0.7 and 1.6. To give the linear layers a non-trivial input size, the
2-D point is multiplied by a fixed random 2×16 matrix `LIFT`. For the shifted
task, the 2-D point is first multiplied by a fixed rotation-and-stretch matrix
`WARP`; the label still comes from the original radius. The cell prints the
fraction of positive labels in each dataset. It should be about one half, so
an accuracy near 50% means the model has learned nothing.

In [2]:
D_IN = 16  # embed the 2-D problem in 16-D so the Linear layers are non-trivial
LIFT = torch.randn(2, D_IN) / 2 ** 0.5          # fixed lift, shared by both tasks
theta = torch.tensor(0.9)                        # the task shift: rotate + stretch
WARP = torch.tensor([[torch.cos(theta), -torch.sin(theta)],
                     [torch.sin(theta), torch.cos(theta)]]) @ \
       torch.tensor([[1.6, 0.0], [0.0, 0.6]])


def make_data(n, shifted, gen):
    xy = torch.randn(n, 2, generator=gen)
    r = xy.norm(dim=1)
    y = ((r > 0.7) & (r < 1.6)).float()          # label: inside the ring?
    if shifted:
        xy = xy @ WARP.T                          # warp the inputs only
    return xy @ LIFT, y


def accuracy(model, x, y):
    with torch.no_grad():
        return ((model(x).squeeze(-1) > 0).float() == y).float().mean().item()


gen = torch.Generator().manual_seed(1)
x_base, y_base = make_data(4000, shifted=False, gen=gen)    # base task, train
x_shift, y_shift = make_data(4000, shifted=True, gen=gen)   # shifted task, train
x_test, y_test = make_data(4000, shifted=True, gen=gen)     # shifted task, test

for name, y in [("base train", y_base), ("shifted train", y_shift),
                ("shifted test", y_test)]:
    print(f"{name:<14} fraction positive = {y.mean().item():.3f}")

base train     fraction positive = 0.498
shifted train  fraction positive = 0.497
shifted test   fraction positive = 0.482


## Pretrain the base model

The base model is a 3-layer MLP with hidden width 256, wide enough that rank 4
is far below full rank. `train` runs 800 full-batch Adam steps on whichever
parameters it is given. We train the base model on the base task and then
measure it on both tasks. Expect high accuracy on the base task and a clear
drop on the shifted task, which shows that adaptation is needed.

In [3]:
H = 256  # hidden width


def make_mlp():
    return nn.Sequential(
        nn.Linear(D_IN, H), nn.ReLU(),
        nn.Linear(H, H), nn.ReLU(),
        nn.Linear(H, 1),
    )


def train(model, params, x, y, steps=800, lr=1e-2):
    opt = torch.optim.Adam(params, lr=lr)
    lossf = nn.BCEWithLogitsLoss()
    for _ in range(steps):
        opt.zero_grad()
        lossf(model(x).squeeze(-1), y).backward()
        opt.step()
    return model


base = make_mlp()
train(base, base.parameters(), x_base, y_base)
base_acc_on_base = accuracy(base, x_base, y_base)
base_acc_on_shift = accuracy(base, x_test, y_test)
print(f"base model on base task:              {base_acc_on_base:6.1%}")
print(f"base model on shifted task (no adapt): {base_acc_on_shift:6.1%}")

base model on base task:              100.0%
base model on shifted task (no adapt):  68.9%


## Reference: full fine-tuning

A copy of the base model is trained on the shifted task with every parameter
trainable, using the same 800 Adam steps. Its test accuracy is the target that
LoRA should match.

In [4]:
full = make_mlp()
full.load_state_dict(base.state_dict())
n_full = sum(p.numel() for p in full.parameters())
train(full, full.parameters(), x_shift, y_shift)
full_acc = accuracy(full, x_test, y_test)
print(f"full fine-tune: {n_full:,} trainable params, shifted-task accuracy {full_acc:.1%}")

full fine-tune: 70,401 trainable params, shifted-task accuracy 99.5%


## The LoRA layer

`LoRALinear` wraps a frozen `nn.Linear` and adds `(alpha/r) * x A^T B^T` to its
output. `A` starts with small random values and `B` with zeros, so the added
term is exactly zero at the start. If both started at zero, the gradient of
each would be proportional to the other, and neither would ever change.
`merge()` adds `(alpha/r) * B @ A` into the frozen weight, so the served model
has the base architecture and no extra inference cost. The cell wraps the base
model with rank 4 and checks that its outputs are unchanged.

In [5]:
class LoRALinear(nn.Module):
    # y = x W^T + b + (alpha/r) * x A^T B^T, with W and b frozen.
    # A: (r, d_in), random; B: (d_out, r), zeros, so the extra term starts at 0.

    def __init__(self, base: nn.Linear, r: int, alpha: float = 16.0):
        super().__init__()
        self.base = base
        for p in self.base.parameters():
            p.requires_grad_(False)                      # freeze W, b
        d_out, d_in = base.weight.shape
        self.A = nn.Parameter(torch.randn(r, d_in) / r)
        self.B = nn.Parameter(torch.zeros(d_out, r))
        self.scale = alpha / r

    def forward(self, x):
        return self.base(x) + self.scale * (x @ self.A.T) @ self.B.T

    def merge(self):
        # fold BA into W: the result is a plain nn.Linear (notes.md §4.1)
        with torch.no_grad():
            self.base.weight += self.scale * self.B @ self.A
        return self.base


def lorafy(model, r):
    # replace every nn.Linear in a Sequential with a LoRA-wrapped version
    return nn.Sequential(*[
        LoRALinear(m, r) if isinstance(m, nn.Linear) else m for m in model
    ])


def n_trainable(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


R = 4
lora = lorafy(base, r=R)
n_lora = n_trainable(lora)
with torch.no_grad():
    gap = (lora(x_test) - base(x_test)).abs().max().item()
print(f"LoRA (r={R}): {n_lora:,} trainable params")
print(f"max |LoRA-wrapped output - base output| before training = {gap}")
assert gap == 0.0, "B = 0 should make the wrapped model identical to the base"

LoRA (r=4): 4,164 trainable params
max |LoRA-wrapped output - base output| before training = 0.0


## Train only the LoRA parameters

Only `A` and `B` in each layer are passed to the optimizer, with the same 800
Adam steps as the full fine-tune. The first assert confirms the base weights
are frozen. Expect a shifted-task accuracy within a few points of full
fine-tuning.

In [6]:
assert all(not p.requires_grad for p in base.parameters())
train(lora, [p for p in lora.parameters() if p.requires_grad], x_shift, y_shift)
lora_acc = accuracy(lora, x_test, y_test)
print(f"LoRA (r={R}): shifted-task accuracy {lora_acc:.1%}  "
      f"(full fine-tune: {full_acc:.1%})")

LoRA (r=4): shifted-task accuracy 99.6%  (full fine-tune: 99.5%)


## Merge the adapters into the weights

Folding `B @ A` into `W` turns each `LoRALinear` back into a plain `nn.Linear`.
The merged network should give the same outputs as the adapted one up to
float32 rounding (a relative difference around 1e-7 to 1e-6, because the
additions happen in a different order), and therefore the same accuracy.

In [7]:
with torch.no_grad():
    out_before = lora(x_test)
merged = nn.Sequential(*[
    m.merge() if isinstance(m, LoRALinear) else m for m in lora
])
with torch.no_grad():
    merge_gap = (merged(x_test) - out_before).abs().max().item()
scale = out_before.abs().max().item()
merged_acc = accuracy(merged, x_test, y_test)
print(f"merged model: {sum(isinstance(m, nn.Linear) for m in merged)} plain Linear layers, "
      f"accuracy {merged_acc:.1%}")
print(f"max |merged output - LoRA output| = {merge_gap:.2e}  "
      f"(largest |output| = {scale:.1f}, relative gap {merge_gap / scale:.1e})")
assert merge_gap / scale < 1e-5, "merge changed the function"

merged model: 3 plain Linear layers, accuracy 99.6%
max |merged output - LoRA output| = 2.44e-04  (largest |output| = 462.7, relative gap 5.3e-07)


## Results and parameter count

The table compares trainable parameters and shifted-task accuracy. For a layer
with input size d_in and output size d_out, LoRA adds `r * (d_in + d_out)`
parameters (`A` has r·d_in, `B` has d_out·r), the same arithmetic as the 7B
count in notes.md §4.1. The asserts at the end are the checks this notebook is
for: LoRA within 3 points of full fine-tuning, and merging leaves accuracy
unchanged.

In [8]:
print("=" * 64)
print("LoRA from scratch: annulus task, shifted inputs")
print("=" * 64)
print(f"base model on base task:              {base_acc_on_base:6.1%}")
print(f"base model on shifted task (frozen):  {base_acc_on_shift:6.1%}")
print("-" * 64)
print(f"{'method':<24}{'trainable params':>18}{'shifted-task acc':>18}")
print(f"{'full fine-tune':<24}{n_full:>18,}{full_acc:>17.1%}")
print(f"{'LoRA (r=' + str(R) + ')':<24}{n_lora:>18,}{lora_acc:>17.1%}")
print(f"{'LoRA merged into W':<24}{0:>18,}{merged_acc:>17.1%}")
print("-" * 64)
print(f"trainable-parameter ratio: {n_lora:,}/{n_full:,} = "
      f"{n_lora / n_full:.1%}  ({n_full / n_lora:.0f}x fewer)")
per_layer = [f"({m.in_features}->{m.out_features}): {R * (m.in_features + m.out_features)}"
             for m in merged if isinstance(m, nn.Linear)]
print("per-layer LoRA params [r*(d_in+d_out)]: " + ", ".join(per_layer))
assert sum(R * (m.in_features + m.out_features)
           for m in merged if isinstance(m, nn.Linear)) == n_lora

assert lora_acc > full_acc - 0.03, "LoRA failed to match full fine-tune"
assert abs(merged_acc - lora_acc) < 1e-6, "merge changed the accuracy"
print(f"\nChecks passed: LoRA is within 3 points of full fine-tuning with "
      f"{n_lora / n_full:.0%} of the trainable params, and merging BA into W "
      "leaves the outputs unchanged.")

LoRA from scratch: annulus task, shifted inputs
base model on base task:              100.0%
base model on shifted task (frozen):   68.9%
----------------------------------------------------------------
method                    trainable params  shifted-task acc
full fine-tune                      70,401            99.5%
LoRA (r=4)                           4,164            99.6%
LoRA merged into W                       0            99.6%
----------------------------------------------------------------
trainable-parameter ratio: 4,164/70,401 = 5.9%  (17x fewer)
per-layer LoRA params [r*(d_in+d_out)]: (16->256): 1088, (256->256): 2048, (256->1): 1028

Checks passed: LoRA is within 3 points of full fine-tuning with 6% of the trainable params, and merging BA into W leaves the outputs unchanged.
